- `Acesso aos dados`: [MCD19A2.061: Terra & Aqua MAIAC Land Aerosol Optical Depth Daily 1km ](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MCD19A2_GRANULES)
- `Período dos dados`: 2000-02-24T00:00:00Z–2024-05-16T23:55:00Z
- `Resolução espacial`: 1000 metros
- `Resolução temporal`: diária
- `Variável utilizada`: Aerosol optical depth over land retrieved in the MODIS Green band (0.55 μm)
- `Código realizado por`: Enrique V. Mattos - 20/05/2024

# **1° Passo:** Preparando ambiente

In [1]:
# instalando bibliotecas
!pip install -q ultraplot cartopy salem rasterio

# iniciando GEE e instalando XEE (transforma dados do GEE para formato DataSet)
!pip install -q eemont
import ee, geemap, eemont
ee.Authenticate()
ee.Initialize(project='ee-enrique', opt_url='https://earthengine-highvolume.googleapis.com')

# importa bibliotecas
import numpy as np
import ultraplot as uplt
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import time
from zipfile import ZipFile
import salem
from datetime import datetime, timedelta
import glob
import xarray as xr
import os
import cartopy.crs as ccrs
import warnings
warnings.filterwarnings('ignore')

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# caminho do drive
dir = '/content/drive/MyDrive/5_EXTENSAO/02_prefeitura_analise_periodo_chuvoso_2024_2025'

# leitura do shapefile do Brasil
shapefile_brasil = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/brasil/BRAZIL.shp')

# leitura do shapefile com a biblioteca SALEM
url = 'https://github.com/evmpython/shapefile/raw/main/'
shp = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
itajuba = salem.read_shapefile(f'{url}itajuba/itajuba.shp')
mg = salem.read_shapefile(f'{url}estado_MG/MG_UF_2019.shp')

# limites do Brasil
lonmin_BR, lonmax_BR, latmin_BR, latmax_BR = -75.0, -33.0, -35.0, 7.0

# limites de MG
lonmin_MG, lonmax_MG, latmin_MG, latmax_MG = -52., -39., -23., -14.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.7/134.7 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.7/224.7 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


/usr/local/lib/python3.11/dist-packages/ultraplot/__init__.py:77: UltraPlotWarning: Rebuilding font cache. This usually happens after installing or updating ultraplot.
  register_fonts(default=True)


Mounted at /content/drive


# **PARTE 1):** Processamento

## 1) Mapa no GEE

In [3]:
#========================================================================================================================#
#                                          FILTRA REGIÃO DE INTERESSE
#========================================================================================================================#
brasil = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
estado_mg = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
municipio_itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))

#========================================================================================================================#
#                                            CARREGA OS DADOS
#========================================================================================================================#
# carrega os dados
AOD_055 = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES') \
            .filter(ee.Filter.date('2024-10-01', '2024-11-01')) \
            .select('Optical_Depth_055') \
            .filterBounds(municipio_itajuba)

#========================================================================================================================#
#                                           PLOTA FIGURA
#========================================================================================================================#
# cria a moldura do mapa
Map = geemap.Map()

# centraliza o mapa na região
Map.centerObject(municipio_itajuba, zoom=11)

# parâmetros de visualização
vis = {'min': 0, 'max': 1, 'palette': ['black', 'blue', 'purple', 'cyan', 'green', 'yellow', 'red']}

# plota mapa
Map.addLayer(AOD_055.max().clip(municipio_itajuba).multiply(0.001), vis, 'AOD MODIS Green band (0.55 μm)')

# contorno da região
style1 = {'color': 'red', 'fillColor': '00000000'}
Map.addLayer(municipio_itajuba.style(**style1), {}, 'MG')

# barra de cores
Map.add_colorbar_branca(colors=vis['palette'], vmin=vis['min'], vmax=vis['max'], layer_name='AOD MODIS Green band (0.55 μm)')

# exibe na tela
Map

Map(center=[-22.420753693113063, -45.41452545652743], controls=(WidgetControl(options=['position', 'transparen…

## 2) Produz série temporal

###Salva arquivos por mês
- Demorou `16.1 s` min para rodar 1 mês para um ponto
- Demorou `18 s` min para rodar 1 mês para um ponto


In [ ]:
%%time
#========================================================================================================================#
#                                          FILTRA REGIÃO DE INTERESSE
#========================================================================================================================#
brasil = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
estado_mg = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', 'Minas Gerais'))
municipio_itajuba = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM2_NAME', 'Itajuba'))
ponto = ee.Geometry.Point([-45.453, -22.4269]).buffer(5000)

#========================================================================================================================#
#                                           DEFINE A DATA INICIAL E FINAL
#========================================================================================================================#
data_inicial = '2019-01-01'
data_final = '2025-04-30'

#========================================================================================================================#
#                                                CARREGA OS DADOS
#========================================================================================================================#
# carrega os dados
AOD_055 = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES') \
            .filter(ee.Filter.date(data_inicial, data_final)) \
            .select('Optical_Depth_055') \
            .filterBounds(ponto)

#========================================================================================================================#
#                                            PRODUZ ARQUIVO NETCDF
#========================================================================================================================#
# Loop entre os meses
for file in pd.date_range(data_inicial.replace('-', ''), data_final.replace('-', ''), freq='1M'):

    #---------------------------------------------------------------------#
    #                      Período dos meses
    #---------------------------------------------------------------------#
    # extrai ano e mês
    ano = file.strftime('%Y')
    mes = file.strftime('%m')

    # monta data no formato "2024-03-31"
    date = f"{ano}-{mes}-{file.strftime('%d')}"

    # extrai o range por intervalo de mês. Exemplo: "DateRange [2024-03-01 00:00:00, 2024-04-01 00:00:00]""
    range = ee.Date(date).getRange('month')

    # calcula a média mensal
    AOD_055_mes = AOD_055.filter(ee.Filter.date(range))

    print('#-------------------------------------------#')
    print('.... PROCESSANDO', f'{ano}-{mes}')
    print('#-------------------------------------------#')

    #---------------------------------------------------------------------#
    #                         Série temporal
    #---------------------------------------------------------------------#
    # gera a serie temporal
    AOD_055_ts = AOD_055_mes.getTimeSeriesByRegion(geometry = municipio_itajuba,
                                                   bands = 'Optical_Depth_055',
                                                   reducer = ee.Reducer.max(),
                                                   scale = 1000)

    # inserindo os dados numa tabela
    AOD_055_ts = geemap.ee_to_df(AOD_055_ts)

    # transformando -9999.0 em NaN
    AOD_055_ts[AOD_055_ts == -9999] = np.nan

    # eliminando os dados NaN
    AOD_055_ts = AOD_055_ts.dropna()

    # aplica fator de escala
    fator_escala = 0.001
    AOD_055_ts['Optical_Depth_055'] = AOD_055_ts['Optical_Depth_055']*fator_escala

    # transformando para DateTime
    AOD_055_ts['date'] = pd.to_datetime(AOD_055_ts['date'], infer_datetime_format = True)

    # transformando a coluna de datas('date') no índice do DataFrame
    AOD_055_ts.index = AOD_055_ts['date']

    # remove colunas
    AOD_055_ts.drop(['reducer', 'date'], inplace=True, axis=1)

    # salva num arquivo CSV
    AOD_055_ts.to_csv(f'{dir}/output/03_AOD/tabela_AOD_055_mensal_{ano}-{mes}.csv')

#-------------------------------------------#
.... PROCESSANDO 2019-01
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-02
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-03
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-04
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-05
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-06
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-07
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2019-08
#-------------------------------------------#
#-------------------------------------------#
.... PROCESSANDO 2

###Concatena os arquivos

In [ ]:
# lista os arquivos
files = sorted(glob.glob(f'{dir}/output/03_AOD/tabela_AOD_055_mensal_*.csv'))

# loop de cada arquivo da lista "files"
df = pd.DataFrame()
for file in files:

    # nome do arquivo
    basename = os.path.basename(os.path.splitext(file)[0])
    print('Processando ===>>>>', basename)

    # leitura da tabela
    df0 = pd.read_csv(file)

    # junta a tabela que foi lida com a anterior
    df = pd.concat([df, df0], ignore_index=True)

# transforma a coluna "data" para o formato "datetime"
df['date'] = pd.to_datetime(df['date'])

# seta a coluna "data" como o índice da tabela
df.set_index('date', inplace=True)

# mostra o dataframe
df

In [ ]:
# agrupa os dados por mês
df_mes = df.groupby(pd.Grouper(freq='1M')).max()['Optical_Depth_055']

# preenche com zeros os meses de 2024 que ainda não chegaram
ano_mes_dia = []
ano_mesi, ano_mesf = '20190101', '20251231'
for data in pd.date_range(ano_mesi, ano_mesf , freq='1M'):
    ano_mes_dia.append(data.strftime('%Y-%m-%d'))
datas = np.array(ano_mes_dia)
df_mes = df_mes.reindex(datas, fill_value=np.nan)

# mostra os dados
display(df_mes)

In [ ]:
# matriz com formato de anos x meses
AOD_055_table = np.reshape(df_mes.values, ((int(ano_mesf[0:4]) - int(ano_mesi[0:4])) + 1, 12), order='C')
AOD_055_table

# **PARTE 2):** Plota figura

In [ ]:
%%time
#========================================================#
#               DEFINIÇÕES INICIAIS
#========================================================#
# moldura da figura
fig, ax = plt.subplots(figsize=(10,5))

# criando heatmap com seaborn
# opções de palltes: https://proplot.readthedocs.io/en/latest/colormaps.html
sns.heatmap(AOD_055_table,
            vmin=0, vmax=1.0,
            cmap='lajolla',
            xticklabels=['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez'],
            yticklabels=uplt.arange(int(ano_mesi[0:4]), int(ano_mesf[0:4]), 1),
            linewidth=0.5,
            linecolor='white',
            cbar_kws={'label': ' ',
                      'shrink': 1.0,
                      'pad': 0.005,
                      'orientation': 'vertical'},
            annot=True, fmt=".2f",
            annot_kws={'color': 'black',
                       'fontsize': 13,
                       'fontweight': 'medium'})

# configurações da barra de cores
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=15, axis="both")
cbar.set_label('Fonte: AQUA e TERRA/Pixel: 5km', fontsize=14)
cbar.ax.minorticks_off()

# título
ax.set_title('AOD 0.55 μm', fontsize=16, color='black', fontweight='bold', loc='left')
ax.text(10.1, -0.092, 'Itajubá (MG)', color='grey', fontsize=16, zorder=5)

# retirar os minorticks
ax.minorticks_off()
plt.grid(False) # Corrected line to turn off the grid

# orientações labels do eixo Y
plt.yticks(rotation=0, fontsize=13)
plt.xticks(rotation=0, fontsize=13)

# informação na figura
ax.text(12.6, 6.6, 'Prof. Enrique Mattos/UNIFEI\ngithub.com/evmpython', fontsize=7, color='black', zorder=5)

# salva figura
plt.tight_layout()
plt.savefig(f'{dir}/output/Fig_5_AOD_055_heatmap_ITAJUBA_2025-04-12_1km_diario.jpg', bbox_inches='tight', dpi=300)
plt.show()